# Phase 3: Build a Strong Retrieval Pipeline

## Step 12: RAG Evaluation

### Learning

- Evaluation datasets
- Retrieval relevance
- Recall@k
- Precision@k
- Hit rate
- Mean Reciprocal Rank
- Answer correctness
- Faithfulness
- Citation accuracy
- Regression testing
- LLM-as-judge evaluation

---

## Key Takeaways

- Retrieval and generation fail independently — a good answer can follow bad
  retrieval (the model got lucky or used background knowledge), and a bad
  answer can follow good retrieval (the right chunk was there, generation
  ignored it). They need separate metrics, checked in that order.
- Retrieval metrics (recall, precision, MRR) don't need a model call — they're
  just set membership and position math against a **ground-truth** list of
  relevant chunk IDs you write by hand. Cheap enough to run on every change.
- "Faithful to the source" and "factually correct" are different checks. An
  answer can be true but not actually supported by what was retrieved — that's
  still a failure, because it means the system got lucky, not right.
- An LLM judge is a tool, not a source of truth — Section 7 exists specifically
  to find out where it disagrees with a human and stop trusting it there.
- A regression isn't "the answer changed" — probabilistic systems always
  change a little. It's "a metric dropped more than it should have," which is
  why a numeric baseline and a threshold matter more than eyeballing outputs.

---

## To do (mirrors the Roadmap 1:1)

1. Create an evaluation dataset
2. Separate retrieval evaluation from answer evaluation
3. Implement retrieval metrics (Hit rate@k, Recall@k, Precision@k, MRR)
4. Add human answer scoring
5. Add faithfulness checks
6. Add an LLM judge
7. Compare human and model evaluation
8. Run configuration experiments
9. Create an evaluation report
10. Add regression tests

Kept simple on purpose: plain functions over lists/dicts, no evaluation
framework or class hierarchy. A little repetition between sections is fine.

## 0. Environment Setup

Same fictional **ByteMage** corpus as Step 11, indexed under its own
index/collection so this notebook runs standalone. One chunk is new:
`legacy-faq-001`, an outdated FAQ that contradicts the current leave policy —
needed for the "contradictory-source" test case in Section 1.

In [1]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

try:
    response = es.info()
    print(response)
except Exception as e:
    print("TYPE:", type(e).__name__)
    print("ERROR:")
    print(e)
    if hasattr(e, "body"):
        print("\nBODY:")
        print(e.body)

import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import config
importlib.reload(config)

{'name': '0e1f447b24d9', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'lHCCr2OyQ0yMLKjGmwdmBg', 'version': {'number': '8.19.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '93788a8c2882eb5b606510680fac214cff1c7a22', 'build_date': '2025-07-23T22:10:18.138212839Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


<module 'config' from '/Users/hirakhan/Developer/AI-ML/rag-chatbot/config.py'>

In [2]:
import json
import time
from typing import Literal

import chromadb
from openai import OpenAI
from pydantic import BaseModel

from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL

client = OpenAI(api_key=OPENAI_API_KEY)
client_chroma = chromadb.HttpClient(host="localhost", port=8000)

INDEX_NAME = "rag_documents_eval"
COLLECTION_NAME = "bytemage_eval_docs"

In [3]:
bytemage_documents = [
    {
        "chunk_id": "company-overview-001",
        "document_id": "company-overview",
        "title": "ByteMage Company Overview",
        "text": (
            "ByteMage was founded in 2015 by Alan Whitfield and Priya Kapoor. "
            "The company is headquartered in Austin, Texas, and builds cloud "
            "applications, data platforms, and AI-powered business tools."
        ),
    },
    {
        "chunk_id": "leave-policy-001",
        "document_id": "leave-policy",
        "title": "ByteMage Leave Policy",
        "text": (
            "ByteMage employees may take up to five sick days per month without "
            "additional approval. Extended sick leave beyond five days requires "
            "notifying HR within 48 hours and is approved by the employee's "
            "direct manager. Unused sick days do not roll over to the next month "
            "and are forfeited at month end."
        ),
    },
    {
        "chunk_id": "legacy-faq-001",
        "document_id": "legacy-faq",
        "title": "ByteMage Legacy FAQ (outdated)",
        "text": (
            "According to an older internal FAQ, ByteMage employees receive "
            "three sick days per month. This FAQ has not been updated since "
            "the leave policy changed."
        ),
    },
    {
        "chunk_id": "compensation-policy-001",
        "document_id": "compensation-policy",
        "title": "ByteMage Compensation Policy",
        "text": (
            "ByteMage salary bands are reviewed every March. Senior Software "
            "Engineers in the AI Platform department fall in Band E5."
        ),
    },
    {
        "chunk_id": "data-retention-policy-001",
        "document_id": "data-retention-policy",
        "title": "ByteMage Data Retention Policy",
        "text": (
            "ByteMage retains customer support data for a minimum of seven "
            "years under regulatory requirement RX-118. Data may be deleted "
            "earlier only upon a verified customer request."
        ),
    },
    {
        "chunk_id": "engineering-handbook-001",
        "document_id": "engineering-handbook",
        "title": "ByteMage Engineering Handbook",
        "text": (
            "ByteMage pull requests require at least one approving review from "
            "a senior engineer before merging to main. John Doe co-authored "
            "this standard as part of the AI Platform team's review guidelines."
        ),
    },
    {
        "chunk_id": "product-roadmap-001",
        "document_id": "product-roadmap",
        "title": "ByteMage Product Roadmap",
        "text": (
            "As of Q3 2026, ByteMage's top product priority is launching the "
            "AI Search Platform's new billing dashboard for enterprise "
            "customers."
        ),
    },
    {
        "chunk_id": "onboarding-guide-001",
        "document_id": "onboarding-guide",
        "title": "ByteMage Onboarding Guide",
        "text": (
            "New ByteMage employees complete orientation during their first "
            "week, including IT setup, benefits enrollment, and an "
            "introduction to the AI Search Platform."
        ),
    },
]

print(f"Loaded {len(bytemage_documents)} chunks.")

Loaded 8 chunks.


In [4]:
# ---- Elasticsearch (lexical side) ----
from elasticsearch.helpers import bulk

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

mapping = {
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "document_id": {"type": "keyword"},
            "title": {"type": "text"},
            "text": {"type": "text"},
        }
    }
}
es.indices.create(index=INDEX_NAME, body=mapping)

actions = [
    {"_index": INDEX_NAME, "_id": chunk["chunk_id"], "_source": chunk}
    for chunk in bytemage_documents
]
success, failed = bulk(es, actions)
print("Successfully indexed into Elasticsearch:", success, "| Failed:", failed)

Successfully indexed into Elasticsearch: 8 | Failed: []


In [5]:
# ---- Chroma (semantic side) ----
try:
    client_chroma.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client_chroma.get_or_create_collection(name=COLLECTION_NAME)

ids, texts, embeddings, metadatas = [], [], [], []

for chunk in bytemage_documents:
    embedding_response = client.embeddings.create(model=EMBEDDING_MODEL, input=chunk["text"])
    ids.append(chunk["chunk_id"])
    texts.append(chunk["text"])
    embeddings.append(embedding_response.data[0].embedding)
    metadatas.append({"document_id": chunk["document_id"], "title": chunk["title"]})

collection.add(ids=ids, documents=texts, embeddings=embeddings, metadatas=metadatas)
print(f"Indexed {collection.count()} documents into Chroma.")

Indexed 8 documents into Chroma.


### Basic retrieval

Same plain vector / lexical / hybrid search functions as Step 11, copied here
so this notebook is self-contained.

In [6]:
def get_embedding(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


def vector_search(query_text, top_k=10):
    query_embedding = get_embedding(query_text)
    raw = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    return [{"chunk_id": chunk_id, "text": raw["documents"][0][i]} for i, chunk_id in enumerate(raw["ids"][0])]


def lexical_search(query_text, top_k=10):
    raw = es.search(index=INDEX_NAME, body={"size": top_k, "query": {"match": {"text": query_text}}})
    return [{"chunk_id": hit["_source"]["chunk_id"], "text": hit["_source"]["text"]} for hit in raw["hits"]["hits"]]


def hybrid_search(query_text, top_k=10):
    """Same idea as Step 7: fuse vector + lexical rankings with RRF (k=60)."""
    vector_results = vector_search(query_text, top_k=10)
    lexical_results = lexical_search(query_text, top_k=10)

    scores = {}
    chunks = {}

    for rank, result in enumerate(vector_results, start=1):
        chunk_id = result["chunk_id"]
        chunks[chunk_id] = result["text"]
        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (60 + rank)

    for rank, result in enumerate(lexical_results, start=1):
        chunk_id = result["chunk_id"]
        chunks[chunk_id] = result["text"]
        scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (60 + rank)

    ranked_chunk_ids = sorted(scores, key=scores.get, reverse=True)
    return [{"chunk_id": cid, "text": chunks[cid]} for cid in ranked_chunk_ids[:top_k]]


def generate_answer(question, retrieved_chunks):
    """Plain answer generation from retrieved chunks -- no citation machinery here,
    Step 10 already covers that. This notebook only needs answer TEXT to evaluate."""
    sources_text = "\n\n".join(chunk["text"] for chunk in retrieved_chunks)
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer the question using only the information below. "
                    "If the answer isn't in the information, say you don't know.\n\n"
                    f"{sources_text}"
                ),
            },
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content

In [7]:
for r in hybrid_search("Who founded ByteMage?", top_k=3):
    print(r["chunk_id"], "-", r["text"][:70])

onboarding-guide-001 - New ByteMage employees complete orientation during their first week, i
company-overview-001 - ByteMage was founded in 2015 by Alan Whitfield and Priya Kapoor. The c
legacy-faq-001 - According to an older internal FAQ, ByteMage employees receive three s


## 1. Create an Evaluation Dataset

> Include: direct factual questions, paraphrased questions, exact identifiers,
> multi-document questions, unanswerable questions, contradictory-source
> questions, follow-up questions, multi-part questions.

Each test case has a `question` (what the user asked), a `retrieval_query`
(what to actually search with — for the follow-up case this is the already-
rewritten standalone form; Step 11 covers *how* to produce that, this notebook
just evaluates with it), and a hand-written ground truth: which chunk(s)
actually answer it.

For the unanswerable case, `relevant_chunk_ids` is empty and `answerable` is
`False` — the correct system behavior is refusing to answer, not retrieving
something and guessing.

In [8]:
EVAL_DATASET = [
    {
        "category": "direct_factual",
        "question": "How many sick days are ByteMage employees allowed per month?",
        "retrieval_query": "How many sick days are ByteMage employees allowed per month?",
        "expected_answer": "Five days without additional approval.",
        "relevant_chunk_ids": ["leave-policy-001"],
        "answerable": True,
    },
    {
        "category": "paraphrased",
        "question": "What's the max number of sick days I can take without needing approval?",
        "retrieval_query": "What's the max number of sick days I can take without needing approval?",
        "expected_answer": "Five days without additional approval.",
        "relevant_chunk_ids": ["leave-policy-001"],
        "answerable": True,
    },
    {
        "category": "exact_identifier",
        "question": "What does regulatory requirement RX-118 require?",
        "retrieval_query": "What does regulatory requirement RX-118 require?",
        "expected_answer": "ByteMage must retain customer support data for at least seven years.",
        "relevant_chunk_ids": ["data-retention-policy-001"],
        "answerable": True,
    },
    {
        "category": "multi_document",
        "question": "Who founded ByteMage, and what is the company's current top product priority?",
        "retrieval_query": "Who founded ByteMage, and what is the company's current top product priority?",
        "expected_answer": "Founded by Alan Whitfield and Priya Kapoor; current priority is the AI Search Platform billing dashboard.",
        "relevant_chunk_ids": ["company-overview-001", "product-roadmap-001"],
        "answerable": True,
    },
    {
        "category": "unanswerable",
        "question": "What is ByteMage's stock ticker symbol?",
        "retrieval_query": "What is ByteMage's stock ticker symbol?",
        "expected_answer": "Not available in the provided documents.",
        "relevant_chunk_ids": [],
        "answerable": False,
    },
    {
        "category": "contradictory_source",
        "question": "How many sick days do ByteMage employees get?",
        "retrieval_query": "How many sick days do ByteMage employees get?",
        "expected_answer": "Five days, per the current policy. An outdated FAQ says three, but the current policy is authoritative.",
        "relevant_chunk_ids": ["leave-policy-001", "legacy-faq-001"],
        "answerable": True,
    },
    {
        "category": "follow_up",
        "question": "When was it?",
        "retrieval_query": "When was ByteMage founded?",
        "expected_answer": "2015.",
        "relevant_chunk_ids": ["company-overview-001"],
        "answerable": True,
    },
    {
        "category": "multi_part",
        "question": "What is the leave allowance, who approves extended leave, and what happens to unused leave?",
        "retrieval_query": "What is the leave allowance, who approves extended leave, and what happens to unused leave?",
        "expected_answer": "Five days allowance; the employee's direct manager approves extended leave; unused days are forfeited, not rolled over.",
        "relevant_chunk_ids": ["leave-policy-001"],
        "answerable": True,
    },
]

eval_path = PROJECT_ROOT / "data" / "eval_dataset.json"
eval_path.write_text(json.dumps(EVAL_DATASET, indent=2))
print(f"Saved {len(EVAL_DATASET)} test cases to {eval_path}")

Saved 8 test cases to /Users/hirakhan/Developer/AI-ML/rag-chatbot/data/eval_dataset.json


## 2. Separate Retrieval Evaluation from Answer Evaluation

> For each test case, evaluate retrieval **before** calling the generation
> model. Record: retrieved chunk IDs, their ranking, whether a relevant chunk
> appeared, rank of the first relevant chunk.

This cell only calls `hybrid_search` — no generation call happens here at
all. That's deliberate: if retrieval already failed, there's no point judging
what the model did with bad context.

In [9]:
def evaluate_retrieval(test_case, top_k=10):
    retrieved = hybrid_search(test_case["retrieval_query"], top_k=top_k)
    retrieved_chunk_ids = [r["chunk_id"] for r in retrieved]
    relevant = set(test_case["relevant_chunk_ids"])

    first_relevant_rank = None
    for rank, chunk_id in enumerate(retrieved_chunk_ids, start=1):
        if chunk_id in relevant:
            first_relevant_rank = rank
            break

    return {
        "category": test_case["category"],
        "question": test_case["question"],
        "retrieved_chunk_ids": retrieved_chunk_ids,
        "relevant_chunk_ids": test_case["relevant_chunk_ids"],
        "first_relevant_rank": first_relevant_rank,
    }


retrieval_results = [evaluate_retrieval(test_case) for test_case in EVAL_DATASET]

for r in retrieval_results:
    print(f"[{r['category']:<18}] first relevant chunk at rank {r['first_relevant_rank']}")

[direct_factual    ] first relevant chunk at rank 2
[paraphrased       ] first relevant chunk at rank 1
[exact_identifier  ] first relevant chunk at rank 1
[multi_document    ] first relevant chunk at rank 1
[unanswerable      ] first relevant chunk at rank None
[contradictory_source] first relevant chunk at rank 1
[follow_up         ] first relevant chunk at rank 1
[multi_part        ] first relevant chunk at rank 1


## 3. Implement Retrieval Metrics

> Calculate Hit rate@k, Recall@k, Precision@k, Mean Reciprocal Rank. Start
> with k = 1, 3, 5, 10. Store results by query category.

Four small functions, each taking the same two inputs: the ranked list of
retrieved chunk IDs, and the set of relevant ones. Unanswerable questions have
no relevant chunks, so recall/precision aren't meaningful for them — those get
skipped (see the `if relevant_chunk_ids` guard).

In [10]:
def hit_at_k(retrieved_chunk_ids, relevant_chunk_ids, k):
    return any(chunk_id in relevant_chunk_ids for chunk_id in retrieved_chunk_ids[:k])


def recall_at_k(retrieved_chunk_ids, relevant_chunk_ids, k):
    found = sum(1 for chunk_id in retrieved_chunk_ids[:k] if chunk_id in relevant_chunk_ids)
    return found / len(relevant_chunk_ids)


def precision_at_k(retrieved_chunk_ids, relevant_chunk_ids, k):
    found = sum(1 for chunk_id in retrieved_chunk_ids[:k] if chunk_id in relevant_chunk_ids)
    return found / k


def reciprocal_rank(retrieved_chunk_ids, relevant_chunk_ids):
    for rank, chunk_id in enumerate(retrieved_chunk_ids, start=1):
        if chunk_id in relevant_chunk_ids:
            return 1 / rank
    return 0.0

In [11]:
K_VALUES = [1, 3, 5, 10]

# One row per (answerable) test case, with metrics kept separate per k --
# blending different k values into one "average recall" would hide exactly
# the k-dependent behavior these metrics are meant to show.
retrieval_metrics = []

for r in retrieval_results:
    relevant_chunk_ids = set(r["relevant_chunk_ids"])
    if not relevant_chunk_ids:
        continue  # unanswerable question -- nothing to measure recall/precision against

    row = {"category": r["category"], "mrr": reciprocal_rank(r["retrieved_chunk_ids"], relevant_chunk_ids)}
    for k in K_VALUES:
        row[f"recall@{k}"] = recall_at_k(r["retrieved_chunk_ids"], relevant_chunk_ids, k)
        row[f"precision@{k}"] = precision_at_k(r["retrieved_chunk_ids"], relevant_chunk_ids, k)
    retrieval_metrics.append(row)

print(f"{'category':<18} {'recall@1':>9} {'recall@3':>9} {'recall@5':>9} {'recall@10':>10} {'MRR':>6}")
for row in retrieval_metrics:
    print(
        f"{row['category']:<18} {row['recall@1']:>9.2f} {row['recall@3']:>9.2f} "
        f"{row['recall@5']:>9.2f} {row['recall@10']:>10.2f} {row['mrr']:>6.2f}"
    )

category            recall@1  recall@3  recall@5  recall@10    MRR
direct_factual          0.00      1.00      1.00       1.00   0.50
paraphrased             1.00      1.00      1.00       1.00   1.00
exact_identifier        1.00      1.00      1.00       1.00   1.00
multi_document          0.50      1.00      1.00       1.00   1.00
contradictory_source      0.50      1.00      1.00       1.00   1.00
follow_up               1.00      1.00      1.00       1.00   1.00
multi_part              1.00      1.00      1.00       1.00   1.00


## 4. Add Human Answer Scoring

> Rubric: `0 = incorrect, 1 = partially correct, 2 = correct`. Score
> correctness, completeness, relevance, citation quality, and appropriate
> refusal separately.

A notebook can't put a real human in the loop, so this section shows the
**template** a human reviewer fills in, and one worked example — imagine
someone reading the question, the retrieved sources, and the generated
answer, then filling in the scores below by hand.

In [12]:
def blank_scorecard():
    return {
        "correctness": None,          # 0, 1, or 2
        "completeness": None,         # 0, 1, or 2
        "relevance": None,            # 0, 1, or 2
        "citation_quality": None,     # 0, 1, or 2 (N/A if the answer doesn't cite)
        "appropriate_refusal": None,  # 0, 1, or 2 -- only scored for unanswerable questions
    }


# Generate one real answer to score.
test_case = EVAL_DATASET[0]  # direct_factual: sick day allowance
retrieved = hybrid_search(test_case["retrieval_query"], top_k=5)
generated_answer = generate_answer(test_case["question"], retrieved)

print("QUESTION:", test_case["question"])
print("EXPECTED:", test_case["expected_answer"])
print("GENERATED:", generated_answer)

QUESTION: How many sick days are ByteMage employees allowed per month?
EXPECTED: Five days without additional approval.
GENERATED: ByteMage employees may take up to five sick days per month without additional approval.


In [13]:
# A human reviewer reads the above and fills this in -- these values are
# filled in by hand, not computed.
human_scorecard = blank_scorecard()
human_scorecard["correctness"] = 2       # matches "five days" exactly
human_scorecard["completeness"] = 2      # states the number and the no-approval-needed condition
human_scorecard["relevance"] = 2         # directly answers what was asked
human_scorecard["citation_quality"] = None  # this notebook's generate_answer() doesn't cite -- N/A
human_scorecard["appropriate_refusal"] = None  # not an unanswerable question -- N/A

print(human_scorecard)

{'correctness': 2, 'completeness': 2, 'relevance': 2, 'citation_quality': None, 'appropriate_refusal': None}


## 5. Add Faithfulness Checks

> Ask an evaluator whether each factual claim is supported by the sources.
> Distinguish: correct but unsupported, supported but incomplete, unsupported,
> contradicted by the source.

This checks something different from correctness: not "is the answer true?"
but "does the retrieved text actually say this?" An answer can be true and
still fail this check, if it happened to be right without the source backing
it up.

In [14]:
class FaithfulnessCheck(BaseModel):
    verdict: Literal["supported", "correct_but_unsupported", "supported_but_incomplete", "unsupported", "contradicted_by_source"]
    reason: str


def check_faithfulness(generated_answer, retrieved_chunks):
    sources_text = "\n\n".join(chunk["text"] for chunk in retrieved_chunks)

    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "Judge whether the ANSWER is faithful to the SOURCES -- i.e. whether "
                    "the sources actually say what the answer claims. Judge only against "
                    "the given sources, not outside knowledge."
                ),
            },
            {"role": "user", "content": f"SOURCES:\n{sources_text}\n\nANSWER:\n{generated_answer}"},
        ],
        response_format=FaithfulnessCheck,
    )
    return response.choices[0].message.parsed


faithfulness = check_faithfulness(generated_answer, retrieved)
print("Verdict:", faithfulness.verdict)
print("Reason: ", faithfulness.reason)

Verdict: supported
Reason:  The sources explicitly state that ByteMage employees may take up to five sick days per month without additional approval.


## 6. Add an LLM Judge

> Strict evaluation prompt with: user question, expected answer, generated
> answer, retrieved sources, scoring rubric. Structured output. The judge
> should not see hidden implementation details it wouldn't normally have.

Same 0/1/2 rubric as Section 4's human scorecard, so the two are directly
comparable in Section 7. The judge only sees what a real evaluator would have
access to -- question, expected answer, generated answer, retrieved source
text. No internal chunk IDs, prompts, or scores from other sections.

In [15]:
class JudgeScore(BaseModel):
    correctness: Literal[0, 1, 2]
    completeness: Literal[0, 1, 2]
    relevance: Literal[0, 1, 2]
    reasoning: str


def llm_judge(question, expected_answer, generated_answer, retrieved_chunks):
    sources_text = "\n\n".join(chunk["text"] for chunk in retrieved_chunks)

    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict evaluator. Score the GENERATED ANSWER against the "
                    "EXPECTED ANSWER and SOURCES, using this rubric for correctness, "
                    "completeness, and relevance: 0 = incorrect, 1 = partially correct, "
                    "2 = correct."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"QUESTION: {question}\n\n"
                    f"EXPECTED ANSWER: {expected_answer}\n\n"
                    f"SOURCES:\n{sources_text}\n\n"
                    f"GENERATED ANSWER: {generated_answer}"
                ),
            },
        ],
        response_format=JudgeScore,
    )
    return response.choices[0].message.parsed


judge_score = llm_judge(test_case["question"], test_case["expected_answer"], generated_answer, retrieved)
print(judge_score)

correctness=2 completeness=2 relevance=2 reasoning='The GENERATED ANSWER correctly states that ByteMage employees may take up to five sick days per month without additional approval, which directly matches the EXPECTED ANSWER in both detail and scope. It includes all relevant information from the SOURCES regarding the sick day policy and excludes unrelated details, making it fully correct, complete, and relevant.'


## 7. Compare Human and Model Evaluation

> Manually score a sample of outputs. Compare with the LLM judge. Identify
> categories where the judge is unreliable.

Just one example here (Sections 4 and 6 scored the same answer) -- in
practice you'd do this across a larger hand-scored sample before trusting the
judge on new data.

In [16]:
print(f"{'':<15} {'human':>6} {'judge':>6}")
print(f"{'correctness':<15} {human_scorecard['correctness']:>6} {judge_score.correctness:>6}")
print(f"{'completeness':<15} {human_scorecard['completeness']:>6} {judge_score.completeness:>6}")
print(f"{'relevance':<15} {human_scorecard['relevance']:>6} {judge_score.relevance:>6}")

if (
    human_scorecard["correctness"] == judge_score.correctness
    and human_scorecard["completeness"] == judge_score.completeness
    and human_scorecard["relevance"] == judge_score.relevance
):
    print("\nHuman and judge agree on this example.")
else:
    print("\nHuman and judge DISAGREE on this example -- inspect before trusting the judge here.")

                 human  judge
correctness          2      2
completeness         2      2
relevance            2      2

Human and judge agree on this example.


## 8. Run Configuration Experiments

> Evaluate combinations such as: different chunk sizes, overlap, top_k,
> vector only, BM25 only, hybrid, hybrid with reranking, query rewriting on
> or off. Store configuration values with each result.

Kept to what this notebook already has: three retrievers (`vector`, `bm25`,
`hybrid`) at two values of `top_k`. Chunk-size/overlap sweeps (Step 9),
reranking (Step 8), and query rewriting (Step 11) plug into this exact same
loop — just swap in whichever function produces `retrieved_chunk_ids` — left
out here to keep this notebook focused on evaluation, not re-running every
prior step.

In [17]:
import time as time_module

RETRIEVERS = {
    "vector": vector_search,
    "bm25": lexical_search,
    "hybrid": hybrid_search,
}

config_results = []

for retriever_name, retriever_fn in RETRIEVERS.items():
    for k in [3, 5]:
        recalls = []
        mrrs = []
        latencies_ms = []

        for test_case in EVAL_DATASET:
            relevant_chunk_ids = set(test_case["relevant_chunk_ids"])
            if not relevant_chunk_ids:
                continue  # skip the unanswerable case -- no ground truth to score against

            t0 = time_module.perf_counter()
            retrieved = retriever_fn(test_case["retrieval_query"], top_k=k)
            latencies_ms.append((time_module.perf_counter() - t0) * 1000)

            retrieved_chunk_ids = [r["chunk_id"] for r in retrieved]
            recalls.append(recall_at_k(retrieved_chunk_ids, relevant_chunk_ids, k))
            mrrs.append(reciprocal_rank(retrieved_chunk_ids, relevant_chunk_ids))

        config_results.append({
            "retriever": retriever_name,
            "top_k": k,
            "avg_recall": sum(recalls) / len(recalls),
            "avg_mrr": sum(mrrs) / len(mrrs),
            "avg_latency_ms": sum(latencies_ms) / len(latencies_ms),
        })

for row in config_results:
    print(row)

{'retriever': 'vector', 'top_k': 3, 'avg_recall': 0.8571428571428571, 'avg_mrr': 0.7857142857142857, 'avg_latency_ms': 398.22741643625443}
{'retriever': 'vector', 'top_k': 5, 'avg_recall': 1.0, 'avg_mrr': 0.8214285714285714, 'avg_latency_ms': 346.10221429362093}
{'retriever': 'bm25', 'top_k': 3, 'avg_recall': 1.0, 'avg_mrr': 1.0, 'avg_latency_ms': 8.967833726533822}
{'retriever': 'bm25', 'top_k': 5, 'avg_recall': 1.0, 'avg_mrr': 1.0, 'avg_latency_ms': 5.2614644269592}
{'retriever': 'hybrid', 'top_k': 3, 'avg_recall': 1.0, 'avg_mrr': 0.9285714285714286, 'avg_latency_ms': 367.0030416937412}
{'retriever': 'hybrid', 'top_k': 5, 'avg_recall': 1.0, 'avg_mrr': 0.9285714285714286, 'avg_latency_ms': 346.1120597203262}


## 9. Create an Evaluation Report

> Table: configuration, retrieval recall, MRR, answer correctness,
> faithfulness, average latency, average cost. Avoid selecting a
> configuration using answer quality alone.

Section 8 already gives recall/MRR/latency per configuration -- print it as a
table. Answer correctness and faithfulness (Sections 4-6) are expensive
(a model call each) and don't vary much by *retriever choice* alone, so in
practice you'd narrow to a couple of finalists using this table first, then
run the human/judge/faithfulness checks only on those -- which is itself the
answer to "avoid selecting a configuration using answer quality alone": pick
finalists on retrieval metrics, confirm with answer quality after.

In [18]:
print(f"{'retriever':<10} {'top_k':>6} {'avg_recall':>11} {'avg_mrr':>9} {'avg_latency_ms':>15}")
for row in config_results:
    print(
        f"{row['retriever']:<10} {row['top_k']:>6} {row['avg_recall']:>11.2f} "
        f"{row['avg_mrr']:>9.2f} {row['avg_latency_ms']:>15.1f}"
    )

retriever   top_k  avg_recall   avg_mrr  avg_latency_ms
vector          3        0.86      0.79           398.2
vector          5        1.00      0.82           346.1
bm25            3        1.00      1.00             9.0
bm25            5        1.00      1.00             5.3
hybrid          3        1.00      0.93           367.0
hybrid          5        1.00      0.93           346.1


## 10. Add Regression Tests

> Choose a baseline configuration. Whenever the system changes: run the
> evaluation dataset, compare against baseline metrics, flag meaningful
> decreases, inspect failed examples, add newly discovered failures to the
> dataset.

`hybrid @ top_k=5` becomes the baseline (best recall/MRR from Section 8's
table). `check_regression` just flags any metric that dropped by more than a
threshold -- small fluctuations are expected from a probabilistic system;
this only flags drops big enough to matter.

In [19]:
BASELINE_NAME = "hybrid @ top_k=5"
baseline_metrics = next(row for row in config_results if row["retriever"] == "hybrid" and row["top_k"] == 5)


def check_regression(current_metrics, baseline_metrics, threshold=0.1):
    """Flag any metric that dropped by more than `threshold` vs. baseline.
    Only checks metrics where lower is worse (recall, mrr) -- latency going
    up is a separate, non-quality regression you'd track alongside this."""
    regressions = []
    for metric_name in ["avg_recall", "avg_mrr"]:
        drop = baseline_metrics[metric_name] - current_metrics[metric_name]
        if drop > threshold:
            regressions.append((metric_name, baseline_metrics[metric_name], current_metrics[metric_name]))
    return regressions


# Simulate "the system changed" by comparing against a weaker configuration.
candidate_metrics = next(row for row in config_results if row["retriever"] == "vector" and row["top_k"] == 5)

print("Baseline: ", BASELINE_NAME, baseline_metrics)
print("Candidate:", "vector @ top_k=5", candidate_metrics)

regressions = check_regression(candidate_metrics, baseline_metrics)
if regressions:
    print("\nREGRESSIONS FOUND:")
    for metric_name, baseline_value, current_value in regressions:
        print(f"- {metric_name}: {baseline_value:.2f} -> {current_value:.2f}")
else:
    print("\nNo regression beyond threshold.")

Baseline:  hybrid @ top_k=5 {'retriever': 'hybrid', 'top_k': 5, 'avg_recall': 1.0, 'avg_mrr': 0.9285714285714286, 'avg_latency_ms': 346.1120597203262}
Candidate: vector @ top_k=5 {'retriever': 'vector', 'top_k': 5, 'avg_recall': 1.0, 'avg_mrr': 0.8214285714285714, 'avg_latency_ms': 346.10221429362093}

REGRESSIONS FOUND:
- avg_mrr: 0.93 -> 0.82


### Reflection

- **Can an answer be correct while retrieval is poor?** Yes — the model can
  fall back on background knowledge. It's still a system failure worth
  tracking, because it won't hold for facts the model doesn't already know.
- **Can retrieval be correct while generation is wrong?** Yes — Section 5's
  faithfulness check exists for exactly this: the right chunk was retrieved,
  but the answer didn't actually use it correctly.
- **Should retrieval and generation be evaluated separately?** Yes — Section 2
  enforces this by structure: retrieval is scored before any generation call
  happens, so a bad answer's cause (bad retrieval vs. bad generation) is never
  ambiguous.
- **Can one LLM reliably evaluate another?** Sometimes — Section 7 is what
  tells you *which* categories to trust it on, by checking it against a human
  on a sample. Never assume reliability without that check.
- **What counts as a regression here?** Not "the answer changed" (expected in
  a probabilistic system) — Section 10 only flags a metric dropping past a
  threshold, e.g. recall falling more than 0.10 versus baseline.
- **How large should an eval dataset be?** Large enough to cover every
  category that matters (Section 1's eight) with more than one example each —
  this notebook uses one per category for clarity, which is too small to trust
  in practice.
- **Should every category get equal weight?** No — weight by how often that
  category occurs for real users, or how costly a failure in it is.
- **How should unanswerable questions be scored?** Separately, via
  "appropriate refusal" (Section 4) — grading them on correctness/completeness
  doesn't make sense when the right answer is "I don't know."
- **Can average metrics hide failures?** Yes — an average recall can look fine
  while one whole category (e.g. contradictory-source) fails every time.
  That's why Section 3 reports metrics per category, not just one global number.
- **Should latency and cost count as quality?** Yes, alongside correctness —
  Section 9's report keeps them as their own columns rather than folding them
  into an answer-quality score, so a slow, expensive, "slightly better" config
  doesn't win by default.